In [1]:
# %% [markdown]
# # Akuisisi Data Akomodasi Kecamatan Kintamani (OpenStreetMap / Overpass)
# BIFP-243 Akuisisi dan Manajemen Data

# %% Sel 1: Impor & konfigurasi
import hashlib
import json
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

# --- SESUAIKAN IDENTITAS ANDA ---
NIM_MAHASISWA = "2501010088"
NAMA_MAHASISWA = "Ni Kadek Satya Giri Sardhula"
EMAIL_MAHASISWA = f"{NIM_MAHASISWA}@instiki.ac.id"

# --- ENDPOINT & QUERY ---
URL_OVERPASS = "https://overpass-api.de/api/interpreter"
HEADERS = {"User-Agent": f"INSTIKI-BIFP243-Praktikum/1.0 ({EMAIL_MAHASISWA})"}

JENIS_AKOMODASI = "hotel|guest_house|hostel|motel|apartment|chalet|resort"
QUERY_OVERPASS = f"""[out:json][timeout:90];
area["name"="Kintamani"]["boundary"="administrative"]["admin_level"="6"]->.kec;
nwr["tourism"~"^({JENIS_AKOMODASI})$"](area.kec);
out center tags;"""

# Bounding box Kintamani (lat_min, lat_max, lon_min, lon_max)
BBOX = (-8.35, -8.12, 115.28, 115.48)

# Angka pembanding BPS. SESUAIKAN dengan dokumen yang Anda pakai.
TOTAL_BPS = 205
TAHUN_BPS = 2024
SUMBER_BPS = "Kintamani Dalam Angka 2024 (isi nomor tabel/halaman di sini)"

BASE_DIR = Path(f"akomodasi_kintamani_{NIM_MAHASISWA}")
DIR_RAW = BASE_DIR / "data" / "raw"
DIR_INTERIM = BASE_DIR / "data" / "interim"
DIR_DOCS = BASE_DIR / "docs"
DIR_REPORTS = BASE_DIR / "reports"
FOLDER_LIST = [DIR_RAW, DIR_INTERIM, BASE_DIR / "notebooks", DIR_DOCS, DIR_REPORTS]


# %% Sel 2: Buat struktur folder
def siapkan_folder():
    for folder in FOLDER_LIST:
        folder.mkdir(parents=True, exist_ok=True)
    print(f"Folder '{BASE_DIR}' dan seluruh sub-foldernya siap.")


# %% Sel 3: Akuisisi data dari Overpass
def ambil_overpass(query, maks_percobaan=4, jeda=5):
    """Ambil data Overpass dengan retry + exponential backoff."""
    for percobaan in range(1, maks_percobaan + 1):
        try:
            print(f"Mengirim request ke Overpass API (percobaan {percobaan}/{maks_percobaan})...")
            r = requests.post(URL_OVERPASS, data={"data": query},
                              headers=HEADERS, timeout=120)
            if r.status_code == 200:
                data = r.json()
                # Overpass bisa membalas 200 tetapi kosong dengan 'remark' (mis. timeout)
                if "remark" in data and not data.get("elements"):
                    print(f"  Remark dari server: {data['remark']}")
                else:
                    print("Pengambilan data sukses (HTTP 200).")
                    return data
            else:
                print(f"  Status HTTP {r.status_code}.")
        except (requests.Timeout, requests.ConnectionError, ValueError) as e:
            print(f"  Error: {type(e).__name__}")

        if percobaan < maks_percobaan:  # tidak menunggu setelah percobaan terakhir
            print(f"  Menunggu {jeda} detik...")
            time.sleep(jeda)
            jeda *= 2

    raise RuntimeError("Gagal mengambil data dari Overpass API setelah batas percobaan.")


def simpan_raw(raw_data):
    ts = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")  # UTC, sama dengan provenance
    path = DIR_RAW / f"osm_akomodasi_kintamani_{ts}.json"
    path.write_text(json.dumps(raw_data, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Raw JSON disimpan: {path}")
    print(f"Total elemen: {len(raw_data.get('elements', []))}")
    return path


# %% Sel 4: Meratakan JSON menjadi tabel
def ratakan(elements):
    """Ratakan JSON Overpass ke DataFrame. Telepon & email sengaja TIDAK diambil (minimisasi data)."""
    baris = []
    for e in elements:
        tg = e.get("tags", {})
        pusat = e.get("center", e)  # way/relation -> center, node -> lat/lon langsung
        baris.append({
            "osm_type": e["type"],
            "osm_id": e["id"],
            "nama": tg.get("name"),
            "jenis": tg.get("tourism"),
            "lat": pusat.get("lat"),
            "lon": pusat.get("lon"),
            "kecamatan": "Kintamani",
            "desa": tg.get("addr:village"),
            "jalan": tg.get("addr:street"),
            "bintang": tg.get("stars"),
            "jumlah_kamar": tg.get("rooms"),
            "website": tg.get("website") or tg.get("contact:website"),
            "url_osm": f"https://www.openstreetmap.org/{e['type']}/{e['id']}",
        })
    return pd.DataFrame(baris)


# %% Sel 5: Pemeriksaan kualitas & pembersihan
def bersihkan(df):
    print("=========== PEMERIKSAAN KUALITAS DATA ===========")
    stats = {"total_awal": len(df)}

    dup = df.duplicated(subset=["osm_type", "osm_id"]).sum()
    print(f"1. Duplikat (osm_type + osm_id): {dup}")
    stats["duplikat"] = int(dup)
    df = df.drop_duplicates(subset=["osm_type", "osm_id"])

    stats["tanpa_nama"] = int(df["nama"].isna().sum())
    print(f"2. Tanpa nama: {stats['tanpa_nama']} (tetap dipertahankan)")

    lat_min, lat_max, lon_min, lon_max = BBOX
    di_dalam = df["lat"].between(lat_min, lat_max) & df["lon"].between(lon_min, lon_max)
    out_bounds = df[~di_dalam]
    stats["di_luar_bbox"] = int(len(out_bounds))
    print(f"3. Koordinat di luar Kintamani: {len(out_bounds)} (DIBUANG dari dataset bersih)")
    if len(out_bounds):
        out_bounds.to_csv(DIR_INTERIM / "akomodasi_di_luar_bbox.csv", index=False, encoding="utf-8")
    df = df[di_dalam].reset_index(drop=True)

    print("\n--- Jumlah per jenis ---")
    print(df["jenis"].value_counts())
    print("=================================================")
    return df, stats


def simpan_interim(df):
    path = DIR_INTERIM / "akomodasi_kintamani.csv"
    df.to_csv(path, index=False, encoding="utf-8")
    print(f"Dataset interim disimpan: {path} ({len(df)} baris)")
    return path


# %% Sel 6: Dokumentasi
def buat_dokumentasi(df, raw_path, raw_data):
    print("Menyusun berkas dokumentasi...")
    total_osm = len(df)
    persentase = round(total_osm / TOTAL_BPS * 100, 2)  # bisa > 100 jika OSM lebih lengkap

    # perbandingan_bps.csv
    pd.DataFrame([{
        "kategori": "TOTAL AKOMODASI",
        "jumlah_osm": total_osm,
        "jumlah_bps": TOTAL_BPS,
        "persentase_kelengkapan": persentase,
        "tahun_data_bps": TAHUN_BPS,
        "sumber_bps": SUMBER_BPS,
        "keterangan": f"OSM = {persentase}% dari angka BPS (>100% berarti OSM memuat lebih banyak objek)",
    }]).to_csv(DIR_DOCS / "perbandingan_bps.csv", index=False, encoding="utf-8")

    # query_overpass.txt
    (DIR_DOCS / "query_overpass.txt").write_text(QUERY_OVERPASS, encoding="utf-8")

    # provenance.json (SHA-256)
    sha256 = hashlib.sha256(raw_path.read_bytes()).hexdigest()
    provenance = {
        "endpoint_url": URL_OVERPASS,
        "query": QUERY_OVERPASS,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "total_elements": len(raw_data.get("elements", [])),
        "total_baris_interim": total_osm,
        "raw_filename": raw_path.name,
        "raw_sha256": sha256,
        "license": "Open Data Commons Open Database License (ODbL)",
        "collector_name": NAMA_MAHASISWA,
        "collector_nim": NIM_MAHASISWA,
    }
    (DIR_DOCS / "provenance.json").write_text(
        json.dumps(provenance, indent=2, ensure_ascii=False), encoding="utf-8")

    # data_dictionary.csv
    data_dict = """kolom,tipe,deskripsi,contoh,boleh_kosong
osm_type,teks,Jenis objek OSM (node/way/relation),way,Tidak
osm_id,bilangan bulat,ID unik objek OpenStreetMap,123456789,Tidak
nama,teks,Nama akomodasi dari tag name,Batur View Guest House,Ya
jenis,teks,Kategori akomodasi dari tag tourism,guest_house,Tidak
lat,desimal,Lintang WGS84 (titik/center),-8.27123,Tidak
lon,desimal,Bujur WGS84 (titik/center),115.37123,Tidak
kecamatan,teks,Kecamatan wilayah query,Kintamani,Tidak
desa,teks,Nama desa dari tag addr:village,Batur Tengah,Ya
jalan,teks,Nama jalan dari tag addr:street,Jl. Raya Penelokan,Ya
bintang,teks,Klasifikasi bintang dari tag stars,3,Ya
jumlah_kamar,teks,Jumlah kamar dari tag rooms,10,Ya
website,teks,URL situs resmi akomodasi,https://baturview.com,Ya
url_osm,teks,Tautan ke objek di openstreetmap.org,https://www.openstreetmap.org/way/123456789,Tidak
"""
    (DIR_DOCS / "data_dictionary.csv").write_text(data_dict, encoding="utf-8")

    # data_ethics_checklist.csv
    ethics = """prinsip_etika,status_kepatuhan,bukti_pelaksanaan
API Limit & Respect,Penuh,User-Agent khusus dan retry dengan exponential backoff (maks 4x)
Minimisasi Data,Penuh,Telepon dan email pemilik tidak diambil dalam skrip
Lisensi & Atribusi,Penuh,Atribusi © OpenStreetMap contributors (ODbL) dicantumkan
Sumber Terlarang,Penuh,Tidak ada scraping Traveloka/Booking/Agoda/Airbnb/Google Maps
Kebenaran Data,Penuh,Raw JSON tidak diubah dan disimpan di data/raw/ dengan hash SHA-256
Transparansi Pembersihan,Penuh,Objek di luar bounding box disimpan terpisah di data/interim/akomodasi_di_luar_bbox.csv
"""
    (DIR_DOCS / "data_ethics_checklist.csv").write_text(ethics, encoding="utf-8")

    # README.md
    readme = f"""# Akuisisi Data Akomodasi di Kecamatan Kintamani

**Mata Kuliah:** BIFP-243 Akuisisi dan Manajemen Data
**Pengambil Data:** {NAMA_MAHASISWA} (NIM: {NIM_MAHASISWA})
**Tanggal Akses (UTC):** {datetime.now(timezone.utc).strftime("%Y-%m-%d")}

## 1. Deskripsi Proyek
Mengumpulkan, meratakan, dan mendokumentasikan data akomodasi di Kecamatan Kintamani, Kabupaten Bangli, Bali secara etis.

## 2. Sumber Data & Atribusi
* **OpenStreetMap (Overpass API)**, berlisensi ODbL.
* **© OpenStreetMap contributors**.

## 3. Batasan Etika
* Mematuhi rate limit Overpass API.
* Minimisasi data: tidak mengambil telepon, email, dan akun media sosial pemilik.
* Tidak melakukan scraping platform OTA (Airbnb, Booking.com, Agoda, dll.).

## 4. Pembersihan Data
* Duplikat (osm_type + osm_id) dibuang.
* Objek di luar bounding box Kintamani dibuang dan disimpan terpisah.
* Objek tanpa nama dipertahankan.

## 5. Cara Menjalankan Ulang
1. Buka di Google Colab atau jalankan `python akuisisi_akomodasi_kintamani.py`.
2. Raw JSON di `data/raw/`, CSV bersih di `data/interim/akomodasi_kintamani.csv`.
3. Grafik dan peta (jika tersedia) di `reports/`.

## 6. Struktur Repositori
- `data/` : raw dan interim
- `docs/` : metadata dan dokumentasi
- `reports/` : laporan, grafik, peta
"""
    (BASE_DIR / "README.md").write_text(readme, encoding="utf-8")
    print("Semua berkas dokumentasi selesai dibuat.")


# %% Sel 7: Sampel verifikasi manual
def sampel_verifikasi(df, n=10):
    df_nama = df.dropna(subset=["nama"])
    if df_nama.empty:
        print("Tidak ada data bernama untuk disampel.")
        return None
    sampel = df_nama.sample(n=min(n, len(df_nama)), random_state=42)
    tabel = sampel[["osm_id", "nama", "jenis", "desa", "lat", "lon", "url_osm"]].reset_index(drop=True)
    tabel.to_csv(DIR_DOCS / "sampel_verifikasi_manual.csv", index=False, encoding="utf-8")
    print(f"=========== {len(tabel)} SAMPEL VERIFIKASI MANUAL ===========")
    try:
        display(tabel)  # noqa: F821  (hanya ada di Jupyter/Colab)
    except NameError:
        print(tabel.to_string())
    return tabel


# %% Sel 8: Grafik & peta (opsional)
def buat_visualisasi(df):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        ax = df["jenis"].value_counts().plot(kind="bar", color="#2a7f62", figsize=(7, 4))
        ax.set_title("Jumlah Akomodasi per Jenis di Kintamani (OSM)")
        ax.set_xlabel("Jenis")
        ax.set_ylabel("Jumlah")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.savefig(DIR_REPORTS / "grafik_jenis_akomodasi.png", dpi=150)
        plt.close()
        print("Grafik disimpan: reports/grafik_jenis_akomodasi.png")
    except ImportError:
        print("matplotlib tidak tersedia, grafik dilewati.")

    try:
        import folium
        peta = folium.Map(location=[df["lat"].mean(), df["lon"].mean()], zoom_start=11)
        for _, r in df.iterrows():
            folium.CircleMarker(
                [r["lat"], r["lon"]], radius=4,
                popup=f"{r['nama'] or '(tanpa nama)'} ({r['jenis']})",
            ).add_to(peta)
        peta.save(str(DIR_REPORTS / "peta_akomodasi.html"))
        print("Peta disimpan: reports/peta_akomodasi.html (© OpenStreetMap contributors)")
    except ImportError:
        print("folium tidak tersedia (pip install folium), peta dilewati.")


# %% Sel 9: Kompres untuk pengumpulan
def buat_zip():
    laporan_pdf = DIR_REPORTS / "laporan_akomodasi_kintamani.pdf"
    if not BASE_DIR.exists():
        print(f"ERROR: folder '{BASE_DIR}' tidak ditemukan.")
        return
    if not laporan_pdf.exists():
        print(f"PERINGATAN: '{laporan_pdf}' belum ada. Masukkan PDF laporan ke folder reports/ "
              "agar tidak ada pengurangan nilai. Zipping tetap dilanjutkan.")
    zip_base = f"akomodasi_kintamani_{NIM_MAHASISWA}"
    hasil = shutil.make_archive(zip_base, "zip", root_dir=".", base_dir=str(BASE_DIR))
    ukuran_mb = Path(hasil).stat().st_size / 1_048_576
    print(f"BERHASIL: '{hasil}' dibuat ({ukuran_mb:.2f} MB).")

    # Tampilkan isi ZIP untuk verifikasi
    import zipfile
    with zipfile.ZipFile(hasil) as z:
        daftar = z.namelist()
    print(f"Isi ZIP ({len(daftar)} entri):")
    for nama in daftar:
        print(f"  - {nama}")

    # Unduh otomatis jika dijalankan di Google Colab
    try:
        from google.colab import files
        files.download(hasil)
        print("Unduhan dimulai. Kumpulkan file ke elsa.instiki.ac.id.")
    except ImportError:
        print(f"Bukan Colab: ambil file '{hasil}' dari folder kerja, lalu kumpulkan ke elsa.instiki.ac.id.")


# %% Sel 9: Laporan otomatis (PDF + Markdown) berisi angka hasil eksekusi
def buat_laporan(df, raw_path, raw_data, stats, tabel_sampel=None):
    """Susun laporan dari data nyata hasil eksekusi. Semua angka dihitung dari df, bukan diketik manual."""
    now = datetime.now(timezone.utc)
    total = len(df)
    pct = round(total / TOTAL_BPS * 100, 2)
    sha = hashlib.sha256(raw_path.read_bytes()).hexdigest()
    per_jenis = df["jenis"].value_counts()
    kol_cek = ["nama", "desa", "jalan", "bintang", "jumlah_kamar", "website"]
    kosong = (df[kol_cek].isna().mean() * 100).round(1)
    grafik = DIR_REPORTS / "grafik_jenis_akomodasi.png"

    if pct >= 100:
        tafsir_bps = (f"Jumlah objek OSM ({total}) setara atau melebihi angka BPS ({TOTAL_BPS}), "
                      f"yaitu {pct}%. Ini tidak otomatis berarti data lebih akurat: perbedaan definisi "
                      "akomodasi, objek yang sudah tutup, atau cakupan waktu dapat menjelaskan selisihnya.")
    else:
        tafsir_bps = (f"Data OSM baru mencakup {pct}% dari angka BPS ({total} dari {TOTAL_BPS} objek), "
                      "sehingga sebagian akomodasi diduga belum dipetakan di OSM atau tidak diberi tag tourism.")

    B = []  # blok neutral: dirender ke Markdown dan PDF
    h1 = lambda t: B.append(("h1", t)); h2 = lambda t: B.append(("h2", t)); h3 = lambda t: B.append(("h3", t))
    p = lambda t: B.append(("p", t)); ul = lambda l: B.append(("ul", l)); tb = lambda r, w=None: B.append(("table", r, w))

    h1("Laporan Akuisisi Data Akomodasi di Kecamatan Kintamani")
    p(f"Mata Kuliah: BIFP-243 Akuisisi dan Manajemen Data | Nama: {NAMA_MAHASISWA} | NIM: {NIM_MAHASISWA} | INSTIKI")
    p(f"Tanggal akses (UTC): {now.strftime('%Y-%m-%d %H:%M')}")

    h2("1. Pendahuluan")
    p("Kecamatan Kintamani, Kabupaten Bangli, Bali, adalah kawasan wisata dengan Gunung dan Danau Batur sebagai "
      "daya tarik utama. Data resmi akomodasi biasanya tersedia dalam bentuk agregat dan terbit dengan jeda waktu. "
      "Laporan ini mendokumentasikan akuisisi data akomodasi dari OpenStreetMap (OSM) secara etis, terstruktur, "
      "dan dapat direproduksi.")

    h2("2. Tujuan")
    ul(["Mengambil data akomodasi Kecamatan Kintamani dari OpenStreetMap melalui Overpass API.",
        "Mengubah JSON hierarkis menjadi dataset tabular yang bersih.",
        "Menilai kualitas data dan membandingkan kelengkapannya dengan angka BPS.",
        "Mendokumentasikan provenance, kamus data, dan kepatuhan etika."])

    h2("3. Sumber Data dan Etika")
    tb([["Aspek", "Penjelasan"],
        ["Sumber", "OpenStreetMap melalui Overpass API (overpass-api.de)"],
        ["Lisensi", "Open Data Commons Open Database License (ODbL)"],
        ["Atribusi", "© OpenStreetMap contributors"],
        ["Pembanding", f"{SUMBER_BPS}, tahun {TAHUN_BPS}"]])
    ul(["Penghormatan API: User-Agent khusus dan retry exponential backoff (maksimum 4 percobaan).",
        "Minimisasi data: telepon, email, dan akun media sosial pemilik tidak diambil.",
        "Tidak ada scraping Traveloka, Booking.com, Agoda, Airbnb, atau Google Maps.",
        "Integritas data mentah: JSON disimpan utuh dan diberi hash SHA-256.",
        "Transparansi pembersihan: objek yang dibuang disimpan terpisah."])

    h2("4. Metodologi")
    h3("4.1 Query Overpass")
    p("Query mengambil node, way, dan relation bertag tourism berikut di area administratif Kintamani "
      f"(admin_level=6): {JENIS_AKOMODASI.replace('|', ', ')}. Perintah out center tags memberi satu titik pusat "
      "untuk objek berbentuk poligon.")
    B.append(("code", QUERY_OVERPASS))
    h3("4.2 Akuisisi dan Penanganan Galat")
    p("Fungsi ambil_overpass() mengulang permintaan saat timeout, galat koneksi, status HTTP selain 200, atau "
      "respons 200 kosong yang memuat remark dari server. Respons mentah disimpan dengan stempel waktu UTC.")
    h3("4.3 Perataan dan Pembersihan")
    ul([f"Data diratakan menjadi tabel berkolom: {', '.join(df.columns)}.",
        "Duplikat dideteksi dari kombinasi osm_type dan osm_id, lalu dibuang.",
        "Objek tanpa nama dipertahankan karena tetap valid sebagai akomodasi.",
        f"Objek di luar bounding box (lintang {BBOX[0]} s.d. {BBOX[1]}; bujur {BBOX[2]} s.d. {BBOX[3]}) "
        "dibuang dan disimpan di akomodasi_di_luar_bbox.csv."])

    h2("5. Hasil")
    h3("5.1 Ringkasan Akuisisi")
    tb([["Indikator", "Nilai"],
        ["Tanggal dan waktu akses (UTC)", now.strftime("%Y-%m-%d %H:%M:%S")],
        ["Total elemen dari Overpass", str(len(raw_data.get("elements", [])))],
        ["Duplikat", str(stats.get("duplikat", 0))],
        ["Objek tanpa nama", str(stats.get("tanpa_nama", 0))],
        ["Objek di luar bounding box (dibuang)", str(stats.get("di_luar_bbox", 0))],
        ["Total baris dataset akhir", str(total)],
        ["Hash SHA-256 data mentah", sha]])

    h3("5.2 Jumlah per Jenis Akomodasi")
    rows = [["Jenis", "Jumlah", "Persentase"]]
    for j in JENIS_AKOMODASI.split("|"):
        n = int(per_jenis.get(j, 0))
        rows.append([j, str(n), f"{n / total * 100:.1f}%" if total else "0%"])
    rows.append(["Total", str(total), "100%"])
    tb(rows)
    if grafik.exists():
        B.append(("img", str(grafik)))

    h3("5.3 Perbandingan dengan Data BPS")
    p("Persentase kelengkapan = (jumlah OSM / jumlah BPS) x 100.")
    tb([["Jumlah OSM", "Jumlah BPS", "Persentase", "Tahun BPS"],
        [str(total), str(TOTAL_BPS), f"{pct}%", str(TAHUN_BPS)]])
    p(tafsir_bps)

    h3("5.4 Kelengkapan Atribut")
    tb([["Kolom", "Persentase kosong"]] + [[k, f"{v}%"] for k, v in kosong.items()])

    h3("5.5 Verifikasi Manual")
    p("Sepuluh sampel acak bernama (random_state=42). Kolom hasil pengecekan diisi manual setelah membuka url_osm.")
    if tabel_sampel is not None and len(tabel_sampel):
        rows = [["No", "Nama", "Jenis", "Desa", "URL OSM", "Hasil cek manual"]]
        for i, r in tabel_sampel.iterrows():
            rows.append([str(i + 1), str(r["nama"]), str(r["jenis"]),
                         "-" if pd.isna(r["desa"]) else str(r["desa"]), str(r["url_osm"]), "(isi setelah dicek)"])
        tb(rows, [1, 4, 2, 3, 6, 3])

    h2("6. Pembahasan")
    terbanyak = per_jenis.index[0] if len(per_jenis) else "-"
    ul([f"Jenis terbanyak adalah {terbanyak} ({int(per_jenis.iloc[0]) if len(per_jenis) else 0} objek).",
        tafsir_bps,
        f"Atribut paling sering kosong: {kosong.idxmax()} ({kosong.max()}%). Atribut opsional di OSM memang "
        "tidak selalu diisi kontributor.",
        "Bounding box berbentuk persegi sehingga objek di tepi wilayah dapat terpotong atau lolos secara keliru."])

    h2("7. Keterbatasan")
    ul(["Data OSM bersifat sukarela sehingga kelengkapannya tidak merata antar-desa.",
        "Akomodasi yang belum dipetakan atau tidak bertag tourism tidak terambil.",
        "Nama area di query bergantung pada penamaan di OSM.",
        "Verifikasi manual hanya mencakup 10 sampel.",
        "Data adalah potret pada waktu akses dan dapat berubah."])

    h2("8. Kesimpulan")
    p(f"Akuisisi data akomodasi Kintamani dari OpenStreetMap berhasil diotomatisasi dengan penanganan galat, "
      f"pemeriksaan kualitas, dan dokumentasi provenance. Dataset akhir berjumlah {total} objek dengan "
      f"kelengkapan {pct}% terhadap data BPS. Data layak dipakai sebagai gambaran awal sebaran akomodasi, "
      "dengan catatan keterbatasan di atas.")

    h2("9. Atribusi dan Daftar Pustaka")
    ul(["OpenStreetMap contributors. Data di bawah lisensi ODbL. https://www.openstreetmap.org/copyright",
        "Overpass API. https://overpass-api.de",
        f"Badan Pusat Statistik. {SUMBER_BPS}."])

    # ---- render Markdown ----
    md = []
    for b in B:
        k = b[0]
        if k == "h1": md.append(f"# {b[1]}\n")
        elif k == "h2": md.append(f"\n## {b[1]}\n")
        elif k == "h3": md.append(f"\n### {b[1]}\n")
        elif k == "p": md.append(b[1] + "\n")
        elif k == "ul": md.append("\n".join(f"- {x}" for x in b[1]) + "\n")
        elif k == "code": md.append("```\n" + b[1] + "\n```\n")
        elif k == "img": md.append(f"![Grafik]({Path(b[1]).name})\n")
        elif k == "table":
            r = b[1]
            md.append("| " + " | ".join(r[0]) + " |\n|" + "---|" * len(r[0]))
            md.extend("| " + " | ".join(x) + " |" for x in r[1:])
            md.append("")
    md_path = DIR_REPORTS / "laporan_akomodasi_kintamani.md"
    md_path.write_text("\n".join(md), encoding="utf-8")
    print(f"Laporan Markdown disimpan: {md_path}")

    # ---- render PDF ----
    pdf_path = DIR_REPORTS / "laporan_akomodasi_kintamani.pdf"
    try:
        from xml.sax.saxutils import escape
        from reportlab.lib import colors
        from reportlab.lib.pagesizes import A4
        from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
        from reportlab.lib.units import cm
        from reportlab.lib.utils import ImageReader
        from reportlab.platypus import (Image, ListFlowable, ListItem, Paragraph,
                                        Preformatted, SimpleDocTemplate, Spacer, Table, TableStyle)

        st = getSampleStyleSheet()
        sel = ParagraphStyle("sel", parent=st["BodyText"], fontSize=8, leading=10)
        selh = ParagraphStyle("selh", parent=sel, textColor=colors.white, fontName="Helvetica-Bold")
        kode = ParagraphStyle("kode", parent=st["Code"], fontSize=7.5, leading=9)
        lebar = A4[0] - 4 * cm
        F = []
        for b in B:
            k = b[0]
            if k == "h1": F += [Paragraph(escape(b[1]), st["Title"]), Spacer(1, 6)]
            elif k == "h2": F += [Spacer(1, 8), Paragraph(escape(b[1]), st["Heading2"])]
            elif k == "h3": F += [Paragraph(escape(b[1]), st["Heading3"])]
            elif k == "p": F += [Paragraph(escape(b[1]), st["BodyText"]), Spacer(1, 4)]
            elif k == "ul":
                F += [ListFlowable([ListItem(Paragraph(escape(x), st["BodyText"])) for x in b[1]],
                                   bulletType="bullet", leftIndent=14), Spacer(1, 4)]
            elif k == "code": F += [Preformatted(b[1], kode), Spacer(1, 6)]
            elif k == "img":
                iw, ih = ImageReader(b[1]).getSize()
                w = min(lebar, 13 * cm)
                F += [Image(b[1], width=w, height=w * ih / iw), Spacer(1, 6)]
            elif k == "table":
                r, w = b[1], b[2]
                data = [[Paragraph(escape(c), selh) for c in r[0]]] + \
                       [[Paragraph(escape(c), sel) for c in row] for row in r[1:]]
                cw = [lebar * x / sum(w) for x in w] if w else None
                t = Table(data, colWidths=cw, repeatRows=1)
                t.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#2a7f62")),
                                       ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
                                       ("VALIGN", (0, 0), (-1, -1), "TOP")]))
                F += [t, Spacer(1, 8)]
        SimpleDocTemplate(str(pdf_path), pagesize=A4, leftMargin=2 * cm, rightMargin=2 * cm,
                          topMargin=2 * cm, bottomMargin=2 * cm,
                          title="Laporan Akomodasi Kintamani", author=NAMA_MAHASISWA).build(F)
        print(f"Laporan PDF disimpan: {pdf_path}")
    except ImportError:
        print("reportlab tidak tersedia (pip install reportlab). Gunakan versi .md dan ekspor sendiri ke PDF.")


# %% Jalankan semuanya
def main():
    siapkan_folder()
    raw_data = ambil_overpass(QUERY_OVERPASS)
    raw_path = simpan_raw(raw_data)

    df = ratakan(raw_data["elements"])
    df, stats = bersihkan(df)
    simpan_interim(df)

    buat_dokumentasi(df, raw_path, raw_data)
    tabel_sampel = sampel_verifikasi(df)
    buat_visualisasi(df)
    buat_laporan(df, raw_path, raw_data, stats, tabel_sampel)
    buat_zip()
    return df


if __name__ == "__main__":
    df_akomodasi = main()
    print(df_akomodasi.head())

Folder 'akomodasi_kintamani_2501010088' dan seluruh sub-foldernya siap.
Mengirim request ke Overpass API (percobaan 1/4)...
Pengambilan data sukses (HTTP 200).
Raw JSON disimpan: akomodasi_kintamani_2501010088/data/raw/osm_akomodasi_kintamani_20261111_090141.json
Total elemen: 32
=========== PEMERIKSAAN KUALITAS DATA ===========
1. Duplikat (osm_type + osm_id): 0
2. Tanpa nama: 1 (tetap dipertahankan)
3. Koordinat di luar Kintamani: 1 (DIBUANG dari dataset bersih)

--- Jumlah per jenis ---
jenis
hotel          16
guest_house     9
hostel          4
chalet          2
Name: count, dtype: int64
Dataset interim disimpan: akomodasi_kintamani_2501010088/data/interim/akomodasi_kintamani.csv (31 baris)
Menyusun berkas dokumentasi...
Semua berkas dokumentasi selesai dibuat.
=========== 10 SAMPEL VERIFIKASI MANUAL ===========


,osm_id,nama,jenis,desa,lat,lon,url_osm
0,11010451459,Manik Tirta Cabin's,hotel,None,-8.249347,115.396872,https://www.openstreetmap.org/node/11010451459
1,6451134585,Caldera,hotel,None,-8.279139,115.367364,https://www.openstreetmap.org/node/6451134585
2,9555392317,Pondok Bali Cottage,guest_house,None,-8.222104,115.398161,https://www.openstreetmap.org/node/9555392317
3,7058054537,Segara Hotel,hotel,None,-8.278086,115.379134,https://www.openstreetmap.org/node/7058054537
4,5065920543,The Volcania Guesthouse,hotel,None,-8.283796,115.365242,https://www.openstreetmap.org/node/5065920543
5,5175562421,Entrance to: Bunbulan Hill Hostel,hostel,None,-8.222014,115.395349,https://www.openstreetmap.org/node/5175562421
6,13068182153,Batur Sunrise Guesthouse,hotel,None,-8.249020,115.400863,https://www.openstreetmap.org/node/13068182153
7,9769685089,Bunbulan Panorama Villas,guest_house,None,-8.222736,115.397971,https://www.openstreetmap.org/node/9769685089
8,5479746121,Hut,chalet,None,-8.242693,115.387587,https://www.openstreetmap.org/node/5479746121
9,1977779907,The Lakeview Hotel Bali,hotel,None,-8.283140,115.365844,https://www.openstreetmap.org/node/1977779907


Grafik disimpan: reports/grafik_jenis_akomodasi.png
Peta disimpan: reports/peta_akomodasi.html (© OpenStreetMap contributors)
Laporan Markdown disimpan: akomodasi_kintamani_2501010088/reports/laporan_akomodasi_kintamani.md
reportlab tidak tersedia (pip install reportlab). Gunakan versi .md dan ekspor sendiri ke PDF.
PERINGATAN: 'akomodasi_kintamani_2501010088/reports/laporan_akomodasi_kintamani.pdf' belum ada. Masukkan PDF laporan ke folder reports/ agar tidak ada pengurangan nilai. Zipping tetap dilanjutkan.
BERHASIL: '/content/akomodasi_kintamani_2501010088.zip' dibuat (0.04 MB).
Isi ZIP (20 entri):
  - akomodasi_kintamani_2501010088/
  - akomodasi_kintamani_2501010088/data/
  - akomodasi_kintamani_2501010088/docs/
  - akomodasi_kintamani_2501010088/notebooks/
  - akomodasi_kintamani_2501010088/reports/
  - akomodasi_kintamani_2501010088/README.md
  - akomodasi_kintamani_2501010088/data/interim/
  - akomodasi_kintamani_2501010088/data/raw/
  - akomodasi_kintamani_2501010088/data/raw/

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Unduhan dimulai. Kumpulkan file ke elsa.instiki.ac.id.
  osm_type      osm_id                     nama        jenis       lat  \
0     node  1977779907  The Lakeview Hotel Bali        hotel -8.283140   
1     node  2107226701                   Arlina  guest_house -8.249815   
2     node  2107227001      Toya Devasya Resort        hotel -8.250164   
3     node  4150326329               Guesthouse        hotel -8.281456   
4     node  4297310696           The Cave Hotel        hotel -8.280748   

          lon  kecamatan  desa jalan bintang jumlah_kamar  \
0  115.365844  Kintamani  None  None    None         None   
1  115.398465  Kintamani  None  None    None         None   
2  115.399798  Kintamani  None  None    None         None   
3  115.362223  Kintamani  None  None    None         None   
4  115.362048  Kintamani  None  None       2            5   

                       website                                        url_osm  
0  http://lakeviewbali.com/en/  https://www.openstree